# Step 3: Projection Model

Builds the per-player-per-week fantasy point projection model. Built up piece by piece, matching `ROADMAP.md` Step 3:
1. Multi-season raw data pull
2. Feature engineering (trailing form, season-to-date/prior-season, categorical)
3. Train/val/test split by season
4. Baseline model
5. LightGBM model
6. Evaluation (per position)


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from nflverse_loader import load_players, load_weekly_stats
from data_loader import normalize_player_week


## 1. Multi-season raw data pull

2019-2024: recent enough that the modern passing-heavy game context still applies, enough volume for weekly-level training. Train 2019-2022, validate 2023, test 2024 (picked in piece 6).

Two real bugs surfaced and fixed while wiring this up (see `PROJECTS.md`/`ROADMAP.md` for detail): `PLAYERS_SCHEMA` was missing `season` (team/bye_week are season-dependent, not identity metadata — only worked before because every caller pulled one season at a time), and 2022's bye-week derivation double-counted the Week 17 Bills-Bengals game (suspended after Damar Hamlin's on-field cardiac arrest, never replayed) as a phantom bye for both teams.

In [2]:
SEASONS = list(range(2019, 2025))

players = load_players(SEASONS)
weekly = load_weekly_stats(SEASONS)
player_week = normalize_player_week(players, weekly)

print(f"players: {players.shape}, weekly: {weekly.shape}, player_week: {player_week.shape}")
player_week["season"].value_counts().sort_index()


players: (18576, 6), weekly: (39224, 14), player_week: (39224, 19)


season
2019    6165
2020    6354
2021    6702
2022    6653
2023    6640
2024    6710
Name: count, dtype: int64

In [3]:
# Sanity check: fantasy points by position, roughly matches expected scale
# (QB highest, then RB/WR, TE a bit lower, K always 0 — kicker scoring isn't
# wired into WEEKLY_STATS_SCHEMA yet, see schema.py).
player_week.groupby("position")["fantasy_points"].agg(["count", "mean"]).round(2)


,count,mean
position,,
DB,29,0.51
DL,1,0.00
K,3330,0.00
LB,9,0.11
P,15,0.00
QB,4065,14.02
RB,9397,8.00
TE,7280,5.63
WR,15098,7.61
